### Tools and Tool Calling in LangChain

Tools are interfaces that allow language models to interact with external systems such as APIs, databases, web search engines, calculators, or custom Python functions.

Rather than relying solely on frozen pre-trained knowledge, a model bound with tools can request external actions to fetch real-time data or perform operations in the real world.

---

### Key Concepts of Tool Calling

* Tool Definition: Functions decorated with @tool or classes inheriting from BaseTool containing type annotations and docstrings.
* Schema Generation: LangChain automatically converts Python function signatures and docstrings into JSON schemas that LLMs understand.
* Model Binding: Attaching tools to a ChatModel using model.bind_tools(tools).
* Tool Invocations: The model returns an AIMessage containing a tool_calls list with tool names and arguments instead of text.
* Tool Execution: Executing the tool function locally and passing the result back as a ToolMessage to complete the conversation loop.

---

### Tools Reference Matrix

| Tool Concept | Description | Code Pattern |
| :--- | :--- | :--- |
| @tool Decorator | Converts a standard Python function into a LangChain tool. | @tool def my_func(a: int) -> str: |
| Pydantic Args Schema | Defines explicit parameter validation and descriptions. | @tool(args_schema=MyArgs) |
| model.bind_tools() | Binds tools to a chat model instance. | model_with_tools = model.bind_tools(tools) |
| AIMessage.tool_calls | Array of tool call requests emitted by the LLM. | response.tool_calls |
| ToolMessage | Message containing tool execution output sent back to LLM. | ToolMessage(content=result, tool_call_id=id) |
| Multiple Tools | Binding multiple distinct tools for model selection. | model.bind_tools([tool1, tool2, tool3]) |
| Parallel Tool Calling | Model requesting multiple tool calls simultaneously. | response.tool_calls returns list of calls |
| API Tools | Wrapping HTTP REST requests inside tool functions. | urllib.request inside @tool function |
| Built-in Tools | Pre-packaged tools from LangChain community. | TavilySearchResults |
| Async Tools | Defining non-blocking async tool functions. | @tool async def my_async_tool(...) |
| Error Handling | Catching tool errors gracefully. | try-except blocks inside tools |
| Return Direct | Returning tool result directly without LLM re-summarization. | @tool(return_direct=True) |
| Tool Choice | Controlling how the model selects tools. | tool_choice="auto", "any", or specific tool || Programmatic Creation | Creating tools from existing functions without decorators. | StructuredTool.from_function() |
| Tool Artifacts | Returning non-text data objects for application logic. | ToolMessage(content=..., artifact=...) |
| Injected Context | Passing hidden session variables directly to tool calls. | config argument in tool function |


### 1. Environment Setup

In [19]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(override=True)

model = init_chat_model("gemini-3.6-flash", model_provider="google_genai")

### 2. Basic Tool Definition with @tool Decorator

The @tool decorator converts a standard Python function into a LangChain tool. Type hints and docstrings are automatically parsed into the tool schema sent to the LLM.

In [10]:
from langchain_core.tools import tool

@tool
def calculate_area(length: float, width: float) -> float:
    """Calculate the area of a rectangle given length and width."""
    return length * width

print("Tool Name:", calculate_area.name)
print("Tool Description:", calculate_area.description)
print("Tool Args Schema:", calculate_area.args)

Tool Name: calculate_area
Tool Description: Calculate the area of a rectangle given length and width.
Tool Args Schema: {'length': {'title': 'Length', 'type': 'number'}, 'width': {'title': 'Width', 'type': 'number'}}


### 3. Pydantic Args Schema

For complex inputs, define a Pydantic BaseModel class to specify explicit field descriptions and validation rules for tool arguments:

In [11]:
from pydantic import BaseModel, Field

class WeatherInput(BaseModel):
    location: str = Field(description="City and state or country, e.g. Tokyo, Japan")
    unit: str = Field(default="celsius", description="Temperature unit: celsius or fahrenheit")

@tool(args_schema=WeatherInput)
def get_current_weather(location: str, unit: str = "celsius") -> str:
    """Get the current weather forecast for a given location."""
    return f"The current weather in {location} is 22 degrees {unit} and sunny."

print("Weather Tool Schema:", get_current_weather.args)

Weather Tool Schema: {'location': {'description': 'City and state or country, e.g. Tokyo, Japan', 'title': 'Location', 'type': 'string'}, 'unit': {'default': 'celsius', 'description': 'Temperature unit: celsius or fahrenheit', 'title': 'Unit', 'type': 'string'}}


### 4. Tool Execution Loop: model.bind_tools and ToolMessage

Attaching tools to a model via model.bind_tools(), inspecting requested tool calls in AIMessage, executing the tool, and returning results via ToolMessage:

In [13]:
from langchain_core.messages import HumanMessage, ToolMessage

model_with_tools = model.bind_tools([calculate_area, get_current_weather])

# Step 1: User prompt
prompt_msg = HumanMessage(content="Calculate the area of a rectangle with length 10 and width 5.")
ai_msg = model_with_tools.invoke([prompt_msg])

print("Model Tool Calls:", ai_msg.tool_calls)

# Step 2: Execute requested tool
tool_call = ai_msg.tool_calls[0]
tool_result = calculate_area.invoke(tool_call["args"])

# Step 3: Return ToolMessage back to model for final response
tool_msg = ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])
final_answer = model_with_tools.invoke([prompt_msg, ai_msg, tool_msg])

print("Final Model Answer:\n", final_answer.content)

Model Tool Calls: [{'name': 'calculate_area', 'args': {'width': 5, 'length': 10}, 'id': 'jSfd98T3', 'type': 'tool_call'}]
Final Model Answer:
 [{'type': 'text', 'text': 'The area of a rectangle with a length of 10 and a width of 5 is 50.', 'extras': {'signature': 'EnsKeQERTTIPccWwlP9Q71ogk+o4gpAHDFryTK7MhKiSVMOwJpRR56yRkWDFqD/gtqdLMLa4Sx10fRHRBjStRTr+WJfmn6BXkXM9KAvonNRp9/WlG9GG6xa/WMeD3x8JVM6VLdQ+5Xx4ABxEHhGGZvzfxCxY5SIMIyXcoXc='}}]


### 5. Multiple Tools Case

Binding multiple different tools (math, weather, user status) so the model dynamically selects the right tool based on prompt context:

In [14]:
@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

@tool
def get_user_status(user_id: int) -> str:
    """Retrieve active status for a given user ID."""
    statuses = {101: "Active", 102: "Pending", 103: "Suspended"}
    return statuses.get(user_id, "User not found")

multi_tool_model = model.bind_tools([calculate_area, add_numbers, get_user_status])

# Prompt targeting user status tool
res1 = multi_tool_model.invoke("What is the status of user 102?")
print("Prompt 1 Tool Requested:", res1.tool_calls[0]["name"])

# Prompt targeting math tool
res2 = multi_tool_model.invoke("Add 45 and 55 together.")
print("Prompt 2 Tool Requested:", res2.tool_calls[0]["name"])

Prompt 1 Tool Requested: get_user_status
Prompt 2 Tool Requested: add_numbers


### 6. Parallel Tool Calling

Modern models can emit multiple tool calls simultaneously in a single prompt to fetch data in parallel:

In [15]:
@tool
def fetch_city_temperature(city: str) -> str:
    """Fetch current temperature for a city."""
    data = {"Tokyo": "18C", "London": "12C", "Paris": "15C", "New York": "20C"}
    return f"{city}: {data.get(city, '22C')}"

parallel_model = model.bind_tools([fetch_city_temperature])

# Prompt requesting info for multiple cities at once
multi_res = parallel_model.invoke("What are the current temperatures in Tokyo, London, and Paris?")

print("Number of Parallel Tool Calls:", len(multi_res.tool_calls))
for call in multi_res.tool_calls:
    print("Call ID:", call["id"], "| Tool:", call["name"], "| Args:", call["args"])

Number of Parallel Tool Calls: 3
Call ID: gU5MZf19 | Tool: fetch_city_temperature | Args: {'city': 'Tokyo'}
Call ID: v0fAv08B | Tool: fetch_city_temperature | Args: {'city': 'London'}
Call ID: pVfHESIS | Tool: fetch_city_temperature | Args: {'city': 'Paris'}


### 7. API Tools (External HTTP Requests)

Wrapping external HTTP API requests (such as fetching JSON from a REST endpoint) inside a LangChain tool:

In [17]:
import urllib.request
import json

@tool
def fetch_github_user(username: str) -> str:
    """Fetch public profile details for a GitHub user by username."""
    url = f"https://api.github.com/users/{username}"
    req = urllib.request.Request(url, headers={"User-Agent": "Python"})
    try:
        with urllib.request.urlopen(req) as response:
            if response.status == 200:
                data = json.loads(response.read().decode())
                return f"GitHub User {data.get('login')}: {data.get('public_repos')} public repos, {data.get('followers')} followers."
    except Exception as e:
        return f"Error fetching GitHub profile: {str(e)}"
    return "Profile not found."

print("API Tool Test:\n", fetch_github_user.invoke({"username": "kapilyadav22"}))

API Tool Test:
 GitHub User kapilyadav22: 31 public repos, 22 followers.


### 8. Built-in Community Tools

LangChain provides pre-packaged integrations for popular services like Wikipedia or Tavily Search:

In [21]:
from langchain_community.tools.tavily_search import TavilySearchResults

# Built-in search tool
web_search = TavilySearchResults(max_results=2)

print("Built-in Tool Name:", web_search.name)
print("Built-in Tool Description:", web_search.description)

Built-in Tool Name: tavily_search_results_json
Built-in Tool Description: A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. Input should be a search query.


### 9. Async Tools (Non-blocking async def)

Defining asynchronous tools using async def for non-blocking asynchronous execution using ainvoke():

In [22]:
import asyncio

@tool
async def async_fetch_stock_price(symbol: str) -> str:
    """Asynchronously fetch stock price ticker symbol."""
    await asyncio.sleep(0.1)  # Simulate non-blocking async network call
    prices = {"AAPL": "$230.50", "GOOGL": "$175.20", "MSFT": "$440.10"}
    return f"{symbol.upper()}: {prices.get(symbol.upper(), '$100.00')}"

# Test async tool using ainvoke
async def test_async_tool():
    result = await async_fetch_stock_price.ainvoke({"symbol": "AAPL"})
    print("Async Tool Result:", result)

await test_async_tool()

Async Tool Result: AAPL: $230.50


### 10. Error Handling in Tools

Tools can handle runtime errors gracefully so exception messages are returned as text to the model instead of crashing execution:

In [23]:
@tool
def safe_divide(a: float, b: float) -> str:
    """Divide a by b with graceful zero-division error handling."""
    try:
        if b == 0:
            return "Error: Division by zero is undefined."
        return str(a / b)
    except Exception as e:
        return f"Execution Error: {str(e)}"

print("Normal Division:", safe_divide.invoke({"a": 10, "b": 2}))
print("Zero Division Error:", safe_divide.invoke({"a": 10, "b": 0}))

Normal Division: 5.0
Zero Division Error: Error: Division by zero is undefined.


### 11. Return Direct (return_direct=True)

Setting return_direct=True on a tool causes the tool execution output to be returned directly to the caller, bypassing the final LLM summary step:

In [24]:
@tool(return_direct=True)
def generate_raw_report(topic: str) -> str:
    """Generate a raw formatted system report directly."""
    return f"[SYSTEM REPORT] Topic: {topic.upper()} | Status: ALL SYSTEMS OPERATIONAL"

print("Return Direct Attribute:", generate_raw_report.return_direct)

direct_model = model.bind_tools([generate_raw_report])
resp = direct_model.invoke("Generate a raw system report for database.")
print("Tool Requested by Model:", resp.tool_calls[0]["name"])

Return Direct Attribute: True
Tool Requested by Model: generate_raw_report


### 12. Tool Choice Control (tool_choice)

The tool_choice parameter controls whether tool calling is automatic, forced, or restricted:

In [26]:
@tool
def lookup_support_ticket(ticket_id: str) -> str:
    """Lookup status of a customer support ticket."""
    return f"Ticket {ticket_id}: Resolved"

# Option A: Automatic tool selection (default)
auto_model = model.bind_tools([lookup_support_ticket], tool_choice="auto")

# Option B: Force model to call AT LEAST ONE tool ("any" or "required")
any_model = model.bind_tools([lookup_support_ticket], tool_choice="any")

# Option C: Force model to call a SPECIFIC tool by name
forced_tool_model = model.bind_tools([lookup_support_ticket], tool_choice="lookup_support_ticket")

forced_res = forced_tool_model.invoke("Hello, I need help.")
print("Forced Tool Call Requested:", forced_res.tool_calls[0]["name"])

Forced Tool Call Requested: lookup_support_ticket


### 13. Programmatic Tool Creation (StructuredTool.from_function)

Instead of using the @tool decorator, you can programmatically create tools from existing Python functions using StructuredTool.from_function():

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

def calculate_discount(price: float, discount_percent: float) -> float:
    return price * (1 - discount_percent / 100)

class DiscountInput(BaseModel):
    price: float = Field(description="Original product price")
    discount_percent: float = Field(description="Discount percentage between 0 and 100")

# Create tool programmatically from function
discount_tool = StructuredTool.from_function(
    func=calculate_discount,
    name="calculate_discount",
    description="Calculate final price after applying a percentage discount.",
    args_schema=DiscountInput
)

print("Programmatic Tool Name:", discount_tool.name)
print("Result:", discount_tool.invoke({"price": 100.0, "discount_percent": 15.0}))

### 14. Tool Call Artifacts (ToolMessage artifact attribute)

ToolMessage supports an artifact parameter to attach non-string data (such as raw DataFrames, binary files, or raw JSON dictionaries) meant for your application logic, while keeping the content field clean for the LLM:

In [ ]:
from langchain_core.messages import ToolMessage

# Simulate a tool that returns a string summary for the LLM and raw data artifact for your application
raw_data_artifact = {"user_id": 101, "login_count": 42, "account_tier": "Enterprise"}

tool_message_with_artifact = ToolMessage(
    content="User 101 status summary: Enterprise tier with 42 logins.",
    artifact=raw_data_artifact,
    tool_call_id="call_999"
)

print("Content sent to LLM:", tool_message_with_artifact.content)
print("Artifact accessed by application:", tool_message_with_artifact.artifact)

### 15. Runtime Injected Arguments and Tool Context

Injected state allows passing contextual variables (such as user_id or session token) directly to a tool at execution time without exposing them to the LLM schema:

In [ ]:
@tool
def fetch_user_dashboard(metrics: str, config: dict = None) -> str:
    """Fetch dashboard metrics for the active session."""
    # Extract injected config or session metadata
    user_id = config.get("configurable", {}).get("user_id", "guest") if config else "guest"
    return f"Dashboard for User {user_id}: Requested metrics '{metrics}' retrieved successfully."

print("Tool Schema (excluding injected config):", fetch_user_dashboard.args)
print("Execution with runtime config:\n", fetch_user_dashboard.invoke(
    {"metrics": "sales, conversion_rate"},
    config={"configurable": {"user_id": "admin_456"}}
))